In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder\
    .appName("Week2_Joins")\
    .master("local[*]")\
    .getOrCreate()

In [4]:
# Employees
emp_data = [
    (1, "Alice", 101),
    (2, "Bob", 102),
    (3, "Charlie", 101),
    (4, "Diana", 103),
    (5, "Eve", 999),      # dept 999 doesn't exist
    (6, "Frank", None),    # no department
]

emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("dept_id", IntegerType(), True),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)

In [5]:
# Departments
dept_data = [
    (101, "Engineering", "Kampala"),
    (102, "Data", "Nairobi"),
    (103, "HR", "Kampala"),
    (104, "Finance", "Nairobi"),  # no employees yet
]

dept_schema = StructType([
    StructField("dept_id", IntegerType(), False),
    StructField("dept_name", StringType(), False),
    StructField("office", StringType(), False),
])

df_dept = spark.createDataFrame(dept_data, dept_schema)

In [6]:
df_emp.join(df_dept, on="dept_id", how="inner").show()

+-------+------+-------+-----------+-------+
|dept_id|emp_id|   name|  dept_name| office|
+-------+------+-------+-----------+-------+
|    101|     1|  Alice|Engineering|Kampala|
|    101|     3|Charlie|Engineering|Kampala|
|    102|     2|    Bob|       Data|Nairobi|
|    103|     4|  Diana|         HR|Kampala|
+-------+------+-------+-----------+-------+



In [8]:
#All 6 employees are here. Eve and Frank have null department info because no match was found. 
#Finance still doesn't appear — we only preserved the left side.
df_emp.join(df_dept, on="dept_id", how="left").show()


+-------+------+-------+-----------+-------+
|dept_id|emp_id|   name|  dept_name| office|
+-------+------+-------+-----------+-------+
|    101|     1|  Alice|Engineering|Kampala|
|    101|     3|Charlie|Engineering|Kampala|
|    102|     2|    Bob|       Data|Nairobi|
|    103|     4|  Diana|         HR|Kampala|
|   NULL|     6|  Frank|       NULL|   NULL|
|    999|     5|    Eve|       NULL|   NULL|
+-------+------+-------+-----------+-------+



In [9]:
df_emp.join(df_dept, on="dept_id", how="right").show()

+-------+------+-------+-----------+-------+
|dept_id|emp_id|   name|  dept_name| office|
+-------+------+-------+-----------+-------+
|    101|     3|Charlie|Engineering|Kampala|
|    101|     1|  Alice|Engineering|Kampala|
|    102|     2|    Bob|       Data|Nairobi|
|    103|     4|  Diana|         HR|Kampala|
|    104|  NULL|   NULL|    Finance|Nairobi|
+-------+------+-------+-----------+-------+



In [10]:
df_emp.join(df_dept, on="dept_id", how="full").show()

+-------+------+-------+-----------+-------+
|dept_id|emp_id|   name|  dept_name| office|
+-------+------+-------+-----------+-------+
|   NULL|     6|  Frank|       NULL|   NULL|
|    101|     3|Charlie|Engineering|Kampala|
|    101|     1|  Alice|Engineering|Kampala|
|    102|     2|    Bob|       Data|Nairobi|
|    103|     4|  Diana|         HR|Kampala|
|    104|  NULL|   NULL|    Finance|Nairobi|
|    999|     5|    Eve|       NULL|   NULL|
+-------+------+-------+-----------+-------+



In [11]:
df_emp.join(df_dept, on="dept_id", how="left_anti").show()

+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|   NULL|     6|Frank|
|    999|     5|  Eve|
+-------+------+-----+



In [13]:
orphans = df_emp.join(df_dept, on="dept_id", how="left_anti")
print(f"Employees with no matching department: {orphans.count()}")
orphans.show()

Employees with no matching department: 2
+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|   NULL|     6|Frank|
|    999|     5|  Eve|
+-------+------+-----+



In [15]:
#SEMI JOIN
df_emp.join(df_dept, on="dept_id", how="left_semi").show()

+-------+------+-------+
|dept_id|emp_id|   name|
+-------+------+-------+
|    101|     1|  Alice|
|    101|     3|Charlie|
|    102|     2|    Bob|
|    103|     4|  Diana|
+-------+------+-------+

